# Code Stylometry and Authorship Verification
This notebook loads the Kaggle dataset, extracts stylometric features (Java-specific syntax and AST structures), and trains an XGBoost model. 

You can run these cells to interactively view the accuracy, classification reports, and generated graphs (Confusion Matrix, ROC Curve, Feature Importance) directly in the notebook!

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_dataset
from src.feature_extraction import extract_features
from src.model import StylometryModel

In [ ]:
# 1. Load the Google Code Jam Dataset
dataset_path = 'dataset/kaggle_data/gpt_java_gcj/dataset'
df = load_dataset(dataset_path)
print(f"\nTotal loaded: {len(df)} files from {df['author'].nunique()} unique authors.")

In [ ]:
# 2. Extract Stylometric Features
# This step parses syntax (brackets, semicolons, keywords) and structural data.
features_df = extract_features(df)
features_df.head()

In [ ]:
# 3. Train the XGBoost Classifier
# The model limits training to the top 50 most prolific authors to speed up evaluation.
model = StylometryModel()
xgb_model, report, accuracy = model.train(features_df, save_plots=True)
print(f"\n🏆 Final Model Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# 4. Visualize Feature Importance
# See which stylometric traits XGBoost relied on most to fingerprint the authors.
importance_scores = xgb_model.feature_importances_
X = features_df.drop(['code', 'author'], axis=1)
feat_imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importance_scores}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df.head(15), palette='viridis')
plt.title('Top 15 Most Important Stylometric Features')
plt.xlabel('XGBoost F-score (Importance)')
plt.ylabel('Stylometric Feature')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Display Evaluation Metrics & Graphs
# The training script automatically saved these graphs to the /results folder.
from IPython.display import Image, display

print("--- Confusion Matrix ---")
display(Image(filename='results/confusion_matrix.png'))

print("--- ROC Curve ---")
display(Image(filename='results/roc_curve.png'))

print("--- Precision-Recall Curve ---")
display(Image(filename='results/pr_curve.png'))